# Framework de Pricing NCO — Implémentation POO

Ce notebook implémente en Python (orienté objet) l'intégralité du framework développé dans le document théorique *"Framework de Pricing NCO — théorie complète"* :

1. **`Bond`** — prix, DV01, Convexité d'une obligation, inversion prix→taux
2. **`BachelierYieldOption`** — call/put sur taux, formules fermées standard
3. **`NCOPricer`** — prix de la NCO (Taylor ordre 2, correction de convexité), Delta/Gamma
4. **`FutureHedgeConverter`** — conversion du delta vers un nominal de future à trader

Chaque classe est testée contre les valeurs numériques déjà vérifiées dans la construction du framework (mêmes chiffres que le document théorique) — si une cellule de test échoue, il y a une régression à corriger avant d'utiliser ce code en production.


In [1]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq
from dataclasses import dataclass, field

np.set_printoptions(suppress=True)


## 1. Classe `Bond`

Encapsule $P(y)$, ses dérivées ($DV01$, Convexité), et l'inversion prix→taux (conversion du strike, section 7 du document théorique).

**Convention d'unités** (cf. glossaire, section 17) : `dv01()` et `convexity()` sont retournés **par bp** et **par bp²** respectivement — cohérent avec un bump de calcul de 1bp exactement.


In [2]:
class Bond:
    '''Obligation à coupon fixe, prix/rendement/sensibilités.

    Convention : `face=100` par défaut (les résultats de `price()` s'entendent
    pour 100 de nominal — à re-scaler par l'appelant si besoin).
    '''

    def __init__(self, coupon: float, maturity: float, freq: int = 1, face: float = 100.0):
        self.coupon = coupon
        self.maturity = maturity
        self.freq = freq
        self.face = face
        self._n = int(round(maturity * freq))
        self._times = np.arange(1, self._n + 1)
        self._cfs = np.full(self._n, coupon / freq * face)
        self._cfs[-1] += face

    def price(self, y) -> np.ndarray:
        '''P(y) — prix pour un taux y (scalaire ou array).'''
        y = np.atleast_1d(np.asarray(y, dtype=float))
        y_per = y[:, None] / self.freq
        disc = 1.0 / (1.0 + y_per) ** self._times[None, :]
        p = (self._cfs[None, :] * disc).sum(axis=1)
        return p if p.size > 1 else p[0]

    def dv01(self, y: float, bump: float = 0.0001) -> float:
        '''DV01 en 'prix par bp' (bump=1bp exactement par défaut).'''
        return -(self.price(y + bump) - self.price(y - bump)) / 2

    def convexity(self, y: float, bump: float = 0.0001) -> float:
        '''Convexité en 'prix par bp^2' (bump=1bp exactement par défaut).'''
        return self.price(y + bump) - 2 * self.price(y) + self.price(y - bump)

    def yield_from_price(self, target_price: float, bracket=(0.0001, 0.20)) -> float:
        '''Inversion P(y)=target_price -> y. C'est la conversion du strike
        (WAP -> y_K), section 7 du document théorique.'''
        return brentq(lambda y: self.price(y) - target_price, *bracket)

    def __repr__(self):
        return f"Bond(coupon={self.coupon:.4%}, maturity={self.maturity}y, freq={self.freq})"


In [3]:
# --- Test de la classe Bond, contre les valeurs déjà vérifiées ---
bond = Bond(coupon=0.03, maturity=9.5)

y_K_ref = 0.0330
WAP = bond.price(y_K_ref)
D = bond.dv01(y_K_ref)
C = bond.convexity(y_K_ref)

print(f"WAP (prix au taux {y_K_ref:.2%})        : {WAP:.4f}")
print(f"DV01                                    : {D:.4f} /bp")
print(f"Convexite                               : {C:.8f} /bp2")

y_K_retrouve = bond.yield_from_price(WAP)
print(f"\nInversion prix->taux : y_K retrouve = {y_K_retrouve:.6f} (attendu {y_K_ref:.6f})")

assert abs(WAP - 97.4797) < 1e-3, "Regression sur le prix !"
assert abs(D - 0.0827) < 1e-3, "Regression sur le DV01 !"
assert abs(C - 0.00008415) < 1e-7, "Regression sur la Convexite !"
assert abs(y_K_retrouve - y_K_ref) < 1e-8, "L'inversion prix->taux ne redonne pas y_K !"
print("\n[OK] Classe Bond validee contre les valeurs de reference.")


WAP (prix au taux 3.30%)        : 97.4797
DV01                                    : 0.0827 /bp
Convexite                               : 0.00008415 /bp2

Inversion prix->taux : y_K retrouve = 0.033000 (attendu 0.033000)

[OK] Classe Bond validee contre les valeurs de reference.


## 2. Classe `BachelierYieldOption`

Formules fermées **standard** de Bachelier (call et put sur taux), dérivées en Partie I du document théorique (section 6). Rappel important : ce sont des objets mathématiques sur le *taux* — le call sur taux correspond à un put sur l'obligation (pas notre NCO), le put sur taux correspond à notre NCO (call sur l'obligation). Voir le tableau de correspondance, section 7.


In [4]:
class BachelierYieldOption:
    '''Call/put sur taux, modele de Bachelier (vol normale).

    F, K en taux (decimal, ex. 0.033 pour 3.30%).
    sigma_N en decimal/an^0.5 (ex. 0.008 pour 80bp).
    T en annees (ex. 1/252 pour un jour ouvre).
    Prix retourne en 'unite de taux' (decimal) -- multiplier par 10000 pour un
    resultat en bp.
    '''

    def __init__(self, F: float, K: float, sigma_N: float, T: float):
        self.F = F
        self.K = K
        self.sigma_N = sigma_N
        self.T = T
        self.s = sigma_N * np.sqrt(T)          # ecart-type total (decimal)
        self.d = (K - F) / self.s               # nombre d'ecarts-types

    def put(self) -> float:
        '''E[max(K-y_T,0)] -- celui dont on a besoin pour la NCO.'''
        d = self.d
        return (self.K - self.F) * norm.cdf(d) + self.s * norm.pdf(d)

    def call(self) -> float:
        '''E[max(y_T-K,0)] -- correspond a un put sur l'obligation, PAS la NCO.'''
        d = self.d
        return (self.F - self.K) * norm.cdf(-d) + self.s * norm.pdf(d)

    def check_put_call_parity(self) -> float:
        '''Doit retourner ~0 : Call - Put - (F-K) = 0.'''
        return self.call() - self.put() - (self.F - self.K)

    def __repr__(self):
        return f"BachelierYieldOption(F={self.F:.4%}, K={self.K:.4%}, sigma_N={self.sigma_N*10000:.0f}bp, d={self.d:.3f})"


In [5]:
# --- Test contre les valeurs deja verifiees dans le document ---
opt = BachelierYieldOption(F=0.0328, K=0.0330, sigma_N=0.008, T=1/252)

put_bp = opt.put() * 10000
call_bp = opt.call() * 10000
parity_check = opt.check_put_call_parity() * 10000

print(f"{opt}")
print(f"Put  : {put_bp:.4f} bp   (attendu 3.1668)")
print(f"Call : {call_bp:.4f} bp   (attendu 1.1668)")
print(f"Call - Put - (F-K) = {parity_check:.6f} bp (doit etre ~0)")

assert abs(put_bp - 3.1668) < 1e-3, "Regression sur le put !"
assert abs(call_bp - 1.1668) < 1e-3, "Regression sur le call !"
assert abs(parity_check) < 1e-6, "La parite call-put est violee !"
print("\n[OK] Classe BachelierYieldOption validee, parite call-put verifiee.")


BachelierYieldOption(F=3.2800%, K=3.3000%, sigma_N=80bp, d=0.397)
Put  : 3.1668 bp   (attendu 3.1668)
Call : 1.1668 bp   (attendu 1.1668)
Call - Put - (F-K) = 0.000000 bp (doit etre ~0)

[OK] Classe BachelierYieldOption validee, parite call-put verifiee.


## 3. Classe `NCOPricer`

Le cœur du framework — Partie III du document théorique. Combine un `Bond`, une hypothèse de vol normale, et le WAP/forward du marché, pour produire :

- le prix **exact au sens du Taylor ordre 2** (terme linéaire de Bachelier + correction de convexité, formule fermée section 10)
- le **garde-fou** (distance de la 2ᵉ racine parasite de la parabole, section 6.2 Partie II)
- **Delta et Gamma**, en espace rendement puis en espace prix (division par $DV01$ et $DV01^2$, section 15)


In [6]:
@dataclass
class NCOPriceResult:
    y_K: float
    F: float
    linear_term_bp: float
    convexity_term_bp: float
    price_bp: float
    garde_fou_std_devs: float
    garde_fou_ok: bool


class NCOPricer:
    '''Price + Greeks de la NCO, Taylor ordre 2 (Partie III du document).

    bond      : instance de Bond (le bond EMIS, celui de la NCO)
    sigma_N   : vol normale (decimal/an^0.5), lue sur VCUB en mode 'Normal'
    T         : maturite en annees (1/252 pour 1 jour ouvre)
    '''

    GARDE_FOU_MIN_STD = 20.0   # seuil mini (en ecarts-types) pour faire confiance au Taylor ordre 2

    def __init__(self, bond: Bond, sigma_N: float, T: float):
        self.bond = bond
        self.sigma_N = sigma_N
        self.T = T
        self.s = sigma_N * np.sqrt(T)          # decimal
        self.s_bp = self.s * 10000

    def _V_of_F(self, y_K: float, D: float, C: float, F: float) -> float:
        '''V(F) -- formule fermee complete (section 10), en bp de prix.'''
        s, s_bp = self.s, self.s_bp
        d = (y_K - F) / s
        linear = D * ((y_K - F) * norm.cdf(d) + s * norm.pdf(d)) * 10000
        convexity_term = 0.5 * C * s_bp**2 * ((d**2 + 1) * norm.cdf(d) + d * norm.pdf(d))
        return linear, convexity_term

    def price(self, WAP: float, F: float) -> NCOPriceResult:
        y_K = self.bond.yield_from_price(WAP)
        D = self.bond.dv01(y_K)
        C = self.bond.convexity(y_K)

        linear_bp, convexity_bp = self._V_of_F(y_K, D, C, F)
        V_bp = linear_bp + convexity_bp

        x2 = 2 * D / C                     # 2e racine de la parabole, en bp
        n_std = x2 / self.s_bp
        garde_fou_ok = n_std > self.GARDE_FOU_MIN_STD

        return NCOPriceResult(
            y_K=y_K, F=F,
            linear_term_bp=linear_bp, convexity_term_bp=convexity_bp,
            price_bp=V_bp, garde_fou_std_devs=n_std, garde_fou_ok=garde_fou_ok,
        )

    def greeks(self, WAP: float, F: float, bump_F: float = 0.0001) -> dict:
        '''Delta et Gamma, en espace rendement puis en espace prix
        (differences finies sur F, conversion par DV01 -- section 15).'''
        y_K = self.bond.yield_from_price(WAP)
        D = self.bond.dv01(y_K)
        C = self.bond.convexity(y_K)

        def V(F_):
            lin, conv = self._V_of_F(y_K, D, C, F_)
            return lin + conv

        delta_yield = -(V(F + bump_F) - V(F - bump_F)) / 2       # prix par bp de F
        gamma_yield = V(F + bump_F) - 2 * V(F) + V(F - bump_F)    # prix par bp^2

        delta_price = delta_yield / D
        gamma_price = gamma_yield / D**2

        return dict(delta_yield=delta_yield, gamma_yield=gamma_yield,
                    delta_price=delta_price, gamma_price=gamma_price)

    def __repr__(self):
        return f"NCOPricer({self.bond}, sigma_N={self.sigma_N*10000:.0f}bp, T={self.T:.4f}y)"


In [7]:
# --- Test contre les valeurs deja verifiees dans le document/la conversation ---
pricer = NCOPricer(bond=bond, sigma_N=0.008, T=1/252)

WAP_market = bond.price(0.0330)   # WAP correspondant a y_K=3.30% (coherent avec tous nos exemples)
F_market = 0.0328                  # forward actuel, legerement ITM

result = pricer.price(WAP=WAP_market, F=F_market)
greeks = pricer.greeks(WAP=WAP_market, F=F_market)

print(f"y_K                          : {result.y_K:.4%}")
print(f"Terme lineaire                : {result.linear_term_bp:.6f}")
print(f"Correction de convexite        : {result.convexity_term_bp:.6f}")
print(f"Prix V                         : {result.price_bp:.6f}")
print(f"Garde-fou : 2e racine a {result.garde_fou_std_devs:.0f} ecarts-types -> {'OK' if result.garde_fou_ok else 'A VERIFIER'}")
print()
print(f"Delta_price                    : {greeks['delta_price']:.4f}")
print(f"Gamma_price                    : {greeks['gamma_price']:.4f}")

assert abs(result.linear_term_bp - 0.261999) < 1e-4, "Regression sur le terme lineaire !"
assert abs(result.convexity_term_bp - 0.000966) < 1e-4, "Regression sur la correction de convexite !"
assert abs(result.price_bp - 0.262965) < 1e-4, "Regression sur le prix final !"
assert abs(greeks['delta_price'] - 0.6565) < 1e-3, "Regression sur Delta_price !"
assert abs(greeks['gamma_price'] - 0.8900) < 1e-2, "Regression sur Gamma_price !"
assert result.garde_fou_ok, "Le garde-fou ne devrait pas se declencher sur cet exemple !"
print("\n[OK] Classe NCOPricer validee contre toutes les valeurs de reference.")


y_K                          : 3.3000%
Terme lineaire                : 0.261999
Correction de convexite        : 0.000966
Prix V                         : 0.262965
Garde-fou : 2e racine a 390 ecarts-types -> OK

Delta_price                    : 0.6565
Gamma_price                    : 0.8900

[OK] Classe NCOPricer validee contre toutes les valeurs de reference.


## 4. Classe `FutureHedgeConverter`

Convertit $\Delta_{price}$ en nominal de future à trader (section 16 du document). Attention à l'unité : `dv01_ctd` doit être exprimé **sur la même base** que le `DV01` du bond émis (même convention `face=100` — voir Point de vigilance section 16 et l'erreur qu'on a explicitement rencontrée puis corrigée en construisant ce framework).


In [8]:
@dataclass
class HedgeResult:
    nominal_bond_eur: float
    nominal_future_eur: float
    n_contracts: float


class FutureHedgeConverter:
    '''Conversion Delta_price -> nominal de future a trader (section 16).'''

    def __init__(self, dv01_bond: float, dv01_ctd: float, cf: float, beta: float,
                 contract_size: float = 100_000):
        '''dv01_bond et dv01_ctd DOIVENT etre sur la meme base de nominal
        (ex. tous les deux 'pour 100 de face') -- sinon le ratio est faux
        d'un facteur d'echelle, silencieusement (cf. document, section 16).'''
        self.dv01_bond = dv01_bond
        self.dv01_ctd = dv01_ctd
        self.cf = cf
        self.beta = beta
        self.contract_size = contract_size

    def convert(self, delta_price: float, n_nco: float) -> HedgeResult:
        nominal_bond = delta_price * n_nco
        ratio = self.dv01_bond / (self.dv01_ctd / self.cf)
        nominal_future = nominal_bond * ratio * self.beta
        n_contracts = nominal_future / self.contract_size
        return HedgeResult(nominal_bond_eur=nominal_bond,
                            nominal_future_eur=nominal_future,
                            n_contracts=n_contracts)


In [9]:
# --- Test contre le pipeline complet deja verifie ---
hedge_converter = FutureHedgeConverter(
    dv01_bond=bond.dv01(0.0330),   # meme base (face=100) que dv01_ctd ci-dessous
    dv01_ctd=0.088,                 # CTD, MEME base de nominal -- point de vigilance
    cf=0.92,
    beta=1.05,
    contract_size=100_000,
)

hedge = hedge_converter.convert(delta_price=greeks['delta_price'], n_nco=10_000_000)

print(f"Nominal hedge bond    : {hedge.nominal_bond_eur:,.0f} EUR")
print(f"Nominal hedge future  : {hedge.nominal_future_eur:,.0f} EUR")
print(f"Nombre de contrats    : {hedge.n_contracts:.1f}")

assert abs(hedge.nominal_bond_eur - 6_565_439) < 1000, "Regression sur le nominal bond !"
assert abs(hedge.n_contracts - 59.6) < 0.5, "Regression sur le nombre de contrats !"
print("\n[OK] Classe FutureHedgeConverter validee.")


Nominal hedge bond    : 6,565,439 EUR
Nominal hedge future  : 5,962,699 EUR
Nombre de contrats    : 59.6

[OK] Classe FutureHedgeConverter validee.


## 5. Pipeline complet, de bout en bout

Reproduit les 6 étapes du document théorique (section 19), à partir de données de marché brutes, jusqu'au nombre de contrats à trader — en une seule fonction, réutilisant les 4 classes ci-dessus.


In [10]:
def price_and_hedge_nco(
    coupon: float, maturity: float,          # caracteristiques du bond emis
    WAP: float,                                # strike (prix), fixe par l'adjudication
    F: float,                                  # forward yield actuel du marche
    sigma_N: float,                            # vol normale (VCUB, mode 'Normal')
    T: float,                                  # maturite de l'option (annees)
    dv01_ctd: float, cf: float, beta: float,   # parametres de conversion vers le future
    n_nco: float,                              # nominal de la NCO
    contract_size: float = 100_000,
) -> dict:
    '''Pipeline complet : donnees de marche -> prix, Greeks, hedge en contrats.'''

    bond_ = Bond(coupon=coupon, maturity=maturity)
    pricer_ = NCOPricer(bond=bond_, sigma_N=sigma_N, T=T)

    result_ = pricer_.price(WAP=WAP, F=F)
    greeks_ = pricer_.greeks(WAP=WAP, F=F)

    if not result_.garde_fou_ok:
        print(f"ATTENTION : garde-fou a seulement {result_.garde_fou_std_devs:.0f} "
              f"ecarts-types (< {NCOPricer.GARDE_FOU_MIN_STD}) -- Taylor ordre 2 fragile ici.")

    converter_ = FutureHedgeConverter(
        dv01_bond=bond_.dv01(result_.y_K), dv01_ctd=dv01_ctd, cf=cf, beta=beta,
        contract_size=contract_size,
    )
    hedge_ = converter_.convert(delta_price=greeks_['delta_price'], n_nco=n_nco)

    return dict(bond=bond_, pricer=pricer_, price_result=result_, greeks=greeks_, hedge=hedge_)


# --- Exemple complet, mêmes données que le document théorique ---
out = price_and_hedge_nco(
    coupon=0.03, maturity=9.5,
    WAP=Bond(0.03, 9.5).price(0.0330),   # strike correspondant a y_K=3.30%
    F=0.0328,
    sigma_N=0.0080, T=1/252,
    dv01_ctd=0.088, cf=0.92, beta=1.05,
    n_nco=10_000_000,
)

print("=" * 55)
print("RAPPORT COMPLET")
print("=" * 55)
r, g, h = out['price_result'], out['greeks'], out['hedge']
print(f"Strike (yield)         : {r.y_K:.4%}")
print(f"Prix NCO (pour 100)     : {r.price_bp:.6f}  (dont convexite: {r.convexity_term_bp:.6f})")
print(f"Garde-fou                : {r.garde_fou_std_devs:.0f} ecarts-types -> {'OK' if r.garde_fou_ok else 'A VERIFIER'}")
print(f"Delta_price               : {g['delta_price']:.4f}")
print(f"Gamma_price                : {g['gamma_price']:.4f}")
print(f"Nominal hedge (bond)        : {h.nominal_bond_eur:,.0f} EUR")
print(f"Nominal hedge (future)      : {h.nominal_future_eur:,.0f} EUR")
print(f"Contrats a trader            : {h.n_contracts:.1f}")


RAPPORT COMPLET
Strike (yield)         : 3.3000%
Prix NCO (pour 100)     : 0.262965  (dont convexite: 0.000966)
Garde-fou                : 390 ecarts-types -> OK
Delta_price               : 0.6565
Gamma_price                : 0.8900
Nominal hedge (bond)        : 6,565,439 EUR
Nominal hedge (future)      : 5,962,699 EUR
Contrats a trader            : 59.6


## Notes d'utilisation

- **`Bond`, `BachelierYieldOption`, `NCOPricer`, `FutureHedgeConverter`** sont indépendantes les unes des autres — tu peux les tester/réutiliser séparément (par exemple, `BachelierYieldOption` seule pour comparer rapidement à un pricer swaption existant).
- **Le garde-fou** (`NCOPricer.GARDE_FOU_MIN_STD`, 20 écarts-types par défaut) est un seuil que j'ai choisi arbitrairement à un niveau très conservateur — nos calculs réels tombent autour de 390 écarts-types. Ajuste-le si besoin, mais ne le supprime pas.
- **Point de vigilance qui a déjà causé une vraie erreur en construisant ce framework** : `dv01_ctd` doit être sur la même base de nominal que `bond.dv01()` — sinon `FutureHedgeConverter` donne un nombre de contrats faux d'un facteur d'échelle, silencieusement (le code tourne, sort un nombre, juste faux).
- Toutes les valeurs de `sigma_N` doivent venir de Bloomberg `VCUB` **en mode "Normal"**, jamais "Black" sans conversion préalable.
